# Titanic Neural Network (PyTorch)

A hand-built feedforward neural network predicting Titanic passenger survival, built in PyTorch to compare against the Scikit-learn Random Forest and Gradient Boosting models in this same project. Includes a diagnosed overfitting bug (model reuse across runs) and its fix (early stopping).

## Version 1: Basic Neural Network

In [ ]:
# --- Version 1: Basic Neural Network (fixed 100 epochs) ---
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score

df = pd.read_csv("train.csv")
df["Age"] = SimpleImputer(strategy="median").fit_transform(df[["Age"]])
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df["Sex"] = LabelEncoder().fit_transform(df["Sex"])
df["Embarked"] = LabelEncoder().fit_transform(df["Embarked"])

features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
X = df[features].values
y = df["Survived"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

class TitanicNet(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.layer1 = nn.Linear(input_size, 16)
        self.layer2 = nn.Linear(16, 8)
        self.output = nn.Linear(8, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.sigmoid(self.output(x))
        return x

model = TitanicNet(input_size=X_train.shape[1])
loss_function = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(100):
    optimizer.zero_grad()
    predictions = model(X_train_t)
    loss = loss_function(predictions, y_train_t)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/100, Loss: {loss.item():.4f}")

model.eval()
with torch.no_grad():
    test_preds = model(X_test_t)
    test_preds_binary = (test_preds >= 0.5).float()

print("V1 Accuracy:", accuracy_score(y_test_t, test_preds_binary))
print("V1 F1-Score:", f1_score(y_test_t, test_preds_binary))

## Version 2: Early-Stopped Neural Network (Overfitting Fix)

In [ ]:
# --- Version 2: Early Stopping (fixes overfitting exposed by tracking test loss) ---
model = TitanicNet(input_size=X_train.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_function = nn.BCELoss()

best_test_loss = float("inf")
patience = 10
patience_counter = 0

for epoch in range(300):
    model.train()
    optimizer.zero_grad()
    predictions = model(X_train_t)
    loss = loss_function(predictions, y_train_t)
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        test_loss = loss_function(model(X_test_t), y_test_t).item()

    if test_loss < best_test_loss:
        best_test_loss = test_loss
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print(f"Stopped at epoch {epoch+1}, best test loss: {best_test_loss:.4f}")
        break

model.load_state_dict(torch.load("best_model.pt"))
model.eval()
with torch.no_grad():
    test_preds = model(X_test_t)
    test_preds_binary = (test_preds >= 0.5).float()

print("V2 (Early Stopped) Accuracy:", accuracy_score(y_test_t, test_preds_binary))
print("V2 (Early Stopped) F1-Score:", f1_score(y_test_t, test_preds_binary))